In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, random, math, re
import numpy as np
import pandas as pd
import ast
from tqdm import tqdm
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import load_npz

In [ ]:
# =============================
# 1) 경로 설정 (환경에 맞게 수정)
# =============================
# 영화 데이터
MOVIES_CSV = "/content/drive/MyDrive/2025Bigdata/data/movie_dataset/movies_edit.csv"
MOVIE_EMB  = "/content/drive/MyDrive/2025Bigdata/hybrid_movie_embeddings.npy"
RATINGS_CSV = "/content/drive/MyDrive/2025Bigdata/data/movie_dataset/ratings.csv"

# 게임 데이터
GAMES_CSV  = "/content/drive/MyDrive/2025Bigdata/data/game_dataset/preprocessed_steam_data.csv"
GAME_EMB   = "/content/drive/MyDrive/2025Bigdata/data/game_dataset/game_embeddings.npz"
GAME_IDS   = "/content/drive/MyDrive/2025Bigdata/data/game_dataset/embedding_appids.csv"

MODEL_SAVE = "/content/drive/MyDrive/2025Bigdata/models/recommend_system.pth"
# =============================
# 2) 환경/장치
# =============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =============================
# 3) 데이터 로드
# =============================
movies = pd.read_csv(MOVIES_CSV)
movie_embs = np.load(MOVIE_EMB)
ratings = pd.read_csv(RATINGS_CSV)

game_embs  = load_npz(GAME_EMB)
game_meta_df = pd.read_csv(GAMES_CSV)
game_ids_df = pd.read_csv(GAME_IDS)

# 임베딩 순서(game_ids_df)에 맞춰 메타데이터(game_meta_df)를 병합
# -> 'games' DataFrame의 0번째 행 = game_embs의 0번째 행
print("Synchronizing game embeddings with metadata...")
games = pd.merge(
    game_ids_df,
    game_meta_df,
    on='appid',
    how='left'
)

print(f"Movie data loaded: {len(movies)} items")
print(f"Game data loaded and synchronized: {len(games)} items")

Device: cpu
Synchronizing game embeddings with metadata...
Movie data loaded: 9742 items
Game data loaded and synchronized: 4507 items


In [ ]:
# =============================
# 4) 기본 전처리: 장르/태그 컬럼 정리 & L2 정규화
# =============================
import ast

def parse_movie_genres(s):
    try:
        return [g.strip().lower() for g in str(s).split("|") if g and g != "(no genres listed)"]
    except:
        return []

def parse_game_genres(s):
    if pd.isna(s) or not s.startswith("["):
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

movies["genre_list"] = movies.get("genres", "").apply(parse_movie_genres)
games["tag_list"]  = games.get("tags", "").apply(parse_game_genres)


# tag_text 컬럼 준비
movies["tag_text"] = movies.get("tags", movies.get("overview", movies.get("title", ""))).fillna("").astype(str)
games["tag_text"]  = games.get("short_description", games.get("name", "")).fillna("").astype(str)

# L2 정규화
movie_embs = normalize(movie_embs, axis=1)
game_embs  = normalize(game_embs, axis=1)

print("movie_embs.shape:", movie_embs.shape, "game_embs.shape:", game_embs.shape)

# =============================
# 5) 장르 매핑 테이블 (영화 장르 -> 게임 장르 후보)
# =============================
genre_mapping = {
    "action": ["action", "shooter", "fighting", "platformer", "fps", "hack and slash"],
    "adventure": ["adventure", "rpg", "puzzle", "exploration", "story-rich", "open world"],
    "comedy": ["casual", "party", "simulation", "funny", "comedy", "relaxing", "cute"],
    "drama": ["visual novel", "narrative", "interactive story", "drama", "emotional", "story rich"],
    "fantasy": ["fantasy", "rpg", "adventure", "magic"],
    "sci-fi": ["sci-fi", "space", "strategy", "cyberpunk", "aliens", "post-apocalyptic"],
    "horror": ["horror", "survival", "thriller", "gore", "jump scare", "dark", "violent"],
    "romance": ["visual novel", "dating sim", "simulation", "emotional", "drama", "story rich", "romance"],
    "thriller": ["horror", "survival", "stealth", "thriller", "mystery", "crime", "psychological"],
    "mystery": ["adventure", "puzzle", "investigation", "mystery", "detective", "crime", "story rich"],
    "biography": ["visual novel", "narrative", "drama", "story rich", "historical", "realistic", "educational"],
    "crime": ["detective", "investigation", "thriller", "crime", "violent"],
    "documentary": ["simulation", "education", "realistic", "historical", "management", "strategy"],
    "music": ["rhythm", "casual", "music", "great soundtrack", "arcade", "relaxing"],
    "war": ["strategy", "simulation", "tactical", "war", "military", "rts", "historical", "realistic"],
    "western": ["action", "shooter", "classic", "survival", "story rich"],
    "animation": ["casual", "visual novel", "anime", "cartoon", "cute", "family", "funny"],
    "family": ["family", "cute", "casual", "puzzle", "relaxing", "educational"],
    "history": ["story rich", "realistic", "world war ii", "world war i", "medieval", "historical"],
    "martial arts": ["ninja", "action", "fighting", "hack and slash"], # 'martial-arts' -> 'martial arts' (영화 장르명 기준)
    "slapstick": ["comedy", "action", "funny", "physics"],
    "sports": ["sports", "football", "basketball", "racing", "pvp", "management"],
    "indie": ["indie"],
}

def movie_genres_to_game_tags(movie_genre_list, mapping):
    import re
    mapped = set()
    for mg in movie_genre_list:
        mg_norm = mg.lower().strip().replace("-", " ").replace("/", " ")
        for key in mapping.keys():
            key_norm = key.lower().strip().replace("-", " ")
            if re.search(rf"\b{re.escape(key_norm)}\b", mg_norm) or key_norm in mg_norm:
                for gg in mapping[key]:
                    mapped.add(gg.lower().strip())
    return mapped



movie_embs.shape: (9742, 1723) game_embs.shape: (4507, 1249)


In [ ]:
# =============================
# 6) TF-IDF 태그 벡터화 (movies+games 통합 corpus)
# =============================
corpus_movies = movies["tag_text"].fillna("").astype(str).tolist()
corpus_games  = games["tag_text"].fillna("").astype(str).tolist()
corpus_all = corpus_movies + corpus_games

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
tfidf_all = tfidf.fit_transform(corpus_all)
movie_tag_vecs = tfidf_all[:len(corpus_movies)]
game_tag_vecs  = tfidf_all[len(corpus_movies):]

# =============================
# 7) Candidate pair 생성 (장르-mapping 기반 -> 태그 유사도 필터)
# =============================
from collections import defaultdict

game_tag_index = defaultdict(list)
for gi, g_row in games.iterrows():
    for tag in g_row["tag_list"]:
        game_tag_index[tag.lower().strip()].append(gi)

pairs = []
THRESH_TAG_SIM = 0.05

print("Generating candidate pairs...")
for mi, m_row in tqdm(movies.iterrows(), total=len(movies)):
    mapped_game_tags = movie_genres_to_game_tags(m_row["genre_list"], genre_mapping)
    if not mapped_game_tags:
        continue
    candidate_set = set()
    for mg in mapped_game_tags:
        candidate_set.update(game_tag_index.get(mg, []))
    if not candidate_set:
        continue
    cand_list = list(candidate_set)
    # compute tag sim for candidates
    movie_vec = movie_tag_vecs[mi]
    game_slice = game_tag_vecs[cand_list]
    sims = cosine_similarity(movie_vec, game_slice).flatten()
    for idx_cand, sim_score in zip(cand_list, sims):
        if sim_score >= THRESH_TAG_SIM:
            # genre overlap count (movie genre in game's genre list)
            overlap_count = 0
            game_actual_tags = set(games.loc[idx_cand, "tag_list"])
            for mapped_tag in mapped_game_tags:
              if mapped_tag in game_actual_tags:
                overlap_count += 1
            tag_score = overlap_count
            combined = 0.7 * (tag_score) + 0.3 * sim_score
            pairs.append((mi, idx_cand, combined))

print("Total candidate pairs:", len(pairs))
# sort and trim
pairs = sorted(pairs, key=lambda x: x[2], reverse=True)
MAX_PAIRS = 200000
pairs = pairs[:MAX_PAIRS]
print("Using pairs:", len(pairs))

# =============================
# 8) train/val split for pairs
# =============================
train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
print("Train pairs:", len(train_pairs), "Val pairs:", len(val_pairs))

# =============================
# 9) Dataset / DataLoader
# =============================
class MovieGamePairDataset(Dataset):
    def __init__(self, pairs, movie_embs, game_embs):
        self.pairs = pairs
        self.movie_embs = movie_embs
        self.game_embs = game_embs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        mi, gi, w = self.pairs[idx]
        mv = self.movie_embs[mi]
        gv = self.game_embs[gi].toarray().flatten()
        return torch.from_numpy(mv).float(), torch.from_numpy(gv).float(), torch.tensor(float(w), dtype=torch.float32)

BATCH_SIZE = 128
train_loader = DataLoader(MovieGamePairDataset(train_pairs, movie_embs, game_embs), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(MovieGamePairDataset(val_pairs, movie_embs, game_embs), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


Generating candidate pairs...


100%|██████████| 9742/9742 [00:13<00:00, 739.12it/s]

Total candidate pairs: 2869
Using pairs: 2869
Train pairs: 2582 Val pairs: 287


In [ ]:
# =============================
# 10) Mapper 모델
# =============================
class MovieToGameMapper(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, output_dim=None, dropout=0.2):
        super().__init__()
        if output_dim is None:
            output_dim = input_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        out = self.net(x)
        out = F.normalize(out, dim=1)  # L2 normalize output
        return out

input_dim = movie_embs.shape[1]
output_dim = game_embs.shape[1]
model = MovieToGameMapper(input_dim=input_dim, hidden_dim=512, output_dim=output_dim, dropout=0.2).to(device)

# =============================
# 11) Loss / Optimizer / Train loop (cosine loss weighted)
# =============================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
ㅁ
def cosine_loss_weighted(pred, target, weight=None):
    # pred, target are expected to be normalized
    cos = (pred * target).sum(dim=1)  # (B,)
    loss_sample = 1.0 - cos
    if weight is not None:
        w = weight / (weight.mean() + 1e-8)
        return (loss_sample * w).mean()
    else:
        return loss_sample.mean()

EPOCHS = 12
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for mb, gb, w in tqdm(train_loader, desc=f"Train Ep{epoch+1}/{EPOCHS}"):
        mb = mb.to(device); gb = gb.to(device); w = w.to(device)
        optimizer.zero_grad()
        out = model(mb)
        gb_n = F.normalize(gb, dim=1)
        loss = cosine_loss_weighted(out, gb_n, weight=w)
        loss.backward()
        optimizer.step()
        running += loss.item() * mb.size(0)
    train_avg = running / len(train_loader.dataset)
    # validation loss
    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for mb, gb, w in val_loader:
            mb = mb.to(device); gb = gb.to(device); w = w.to(device)
            out = model(mb)
            loss_val = cosine_loss_weighted(out, F.normalize(gb, dim=1), weight=w)
            val_running += loss_val.item() * mb.size(0)
    val_avg = val_running / len(val_loader.dataset)
    print(f"[Epoch {epoch+1}] train_loss: {train_avg:.6f}  val_loss: {val_avg:.6f}")

# save model
model_dir = os.path.dirname(MODEL_SAVE)
os.makedirs(model_dir, exist_ok=True)
torch.save(model.state_dict(), MODEL_SAVE)
print("Saved mapper ->", MODEL_SAVE)

# =============================
# 12) helper: map batch (numpy)
# =============================
def map_batch_np(X_np, model, device, batch_size=1024):
    model.eval()
    out_list = []
    for i in range(0, X_np.shape[0], batch_size):
        xb = torch.from_numpy(X_np[i:i+batch_size]).float().to(device)
        with torch.no_grad():
            out = model(xb).cpu().numpy()
        out_list.append(out)
    return np.vstack(out_list)


Train Ep1/12: 100%|██████████| 21/21 [00:01<00:00, 14.56it/s]


[Epoch 1] train_loss: 0.429536  val_loss: 0.167585


Train Ep2/12: 100%|██████████| 21/21 [00:01<00:00, 11.40it/s]


[Epoch 2] train_loss: 0.136709  val_loss: 0.113495


Train Ep3/12: 100%|██████████| 21/21 [00:02<00:00, 10.09it/s]


...

[Epoch 10] train_loss: 0.102428  val_loss: 0.105992


Train Ep11/12: 100%|██████████| 21/21 [00:02<00:00,  7.72it/s]


[Epoch 11] train_loss: 0.101874  val_loss: 0.105991


Train Ep12/12: 100%|██████████| 21/21 [00:01<00:00, 14.28it/s]


[Epoch 12] train_loss: 0.101242  val_loss: 0.106108
Saved mapper -> /content/drive/MyDrive/2025Bigdata/models/recommend_system.pth


In [ ]:
# =============================
# 13) 사용자 벡터 생성: 평점 가중 평균
# =============================
# build movieId -> movie_index mapping if needed
if "movie_index" in ratings.columns:
    # assume this index corresponds to movie_embs index
    pass
else:
    # try map by movieId -> movies.index
    if "movieId" in movies.columns and "movieId" in ratings.columns:
        id2idx = {mid: idx for idx, mid in movies["movieId"].items()}
        ratings["movie_index"] = ratings["movieId"].map(id2idx)
    else:
        raise RuntimeError("ratings.csv must contain movieId or movie_index to map to embeddings.")

def user_weighted_vector(user_id, ratings_df, movie_embs, min_rating=3.5):
    user_df = ratings_df[ratings_df["userId"] == user_id]
    liked = user_df[user_df["rating"] >= min_rating]
    if liked.empty:
        return None
    # drop missing movie_index
    liked = liked.dropna(subset=["movie_index"])
    idxs = liked["movie_index"].astype(int).values
    scores = liked["rating"].values.astype(float)
    embs = movie_embs[idxs]
    weighted = (embs * scores.reshape(-1,1)).sum(axis=0) / (scores.sum() + 1e-8)
    # normalize
    weighted = weighted / (np.linalg.norm(weighted) + 1e-8)
    return weighted

# =============================
# 14) 추천 함수: user->mapped->similarity->TopK
# =============================
def recommend_with_mapper(user_id, ratings_df, movie_embs, game_embs, model, top_k, min_rating=3.5):
    uvec = user_weighted_vector(user_id, ratings_df, movie_embs, min_rating=min_rating)
    if uvec is None:
        return []
    # map through model
    mapped = map_batch_np(uvec.reshape(1,-1), model, device)  # normalized
    # ensure game_embs normalized
    game_norm = normalize(game_embs, axis=1)
    sims = cosine_similarity(mapped, game_norm).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return [(int(i), float(sims[i])) for i in top_idx]


In [ ]:
# =============================
# 15) 추천 예시 (유저 하나)
# =============================
sample_user = ratings["userId"].unique()[0]
print("Sample user:", sample_user)

recs = recommend_with_mapper(sample_user,
                             ratings,
                             movie_embs,
                             game_embs,
                             model,
                             top_k=5,
                             min_rating=4.0)

print("Top recommendations (game_idx, score):", recs)
print("추천 게임 제목 리스트:")
for gi, sc in recs:
  if gi < len(games):
    game_name = games.iloc[gi]["original_name"]
    print(f"{game_name} (Score: {sc:4f})")
  else:
    print(f"Error: Index {gi} out of bounds for games metadata.")

# =============================
# 16) 평가 루틴: Precision@K, Recall@K, NDCG@K for users
#    - ground truth: 사용자가 테스트셋에서 rating >= threshold 로 좋아한 게임들
#    - NOTE: 이 데이터셋에는 사용자-게임 상호작용이 없으므로 우리는 "proxy" 평가 사용:
#      validation pairs 포함된 영화->게임 대응을 이용한 recall@K (mapping 품질)
#    - 추가: 실제 게임 좋아요 데이터가 있다면 사용자 기반 평가로 바꾸세요.
# =============================
# Evaluate mapping quality on val_pairs (recall@K) - already computed below as sanity check
#VAL_N = min(2000, len(val_pairs))
#val_pairs_small = val_pairs[:VAL_N]
#val_movie_idx = [p[0] for p in val_pairs_small]
#val_game_idx  = [p[1] for p in val_pairs_small]
#val_movie_embs = movie_embs[val_movie_idx]
#mapped_val = map_batch_np(val_movie_embs, model, device, batch_size=512)
#game_embs_norm = normalize(game_embs, axis=1)
#sims_mat = cosine_similarity(mapped_val, game_embs_norm)
#K_list = [5,10,20]
#for K in K_list:
    #hits = 0
    #for i, true_g in enumerate(val_game_idx):
        #topk = sims_mat[i].argsort()[::-1][:K]
        #if true_g in topk:
            #hits += 1
    #recall = hits / len(val_pairs_small)
    #print(f"Recall@{K}: {recall:.4f}")

# =============================
# 17) (선택) 전체 사용자 평가 예시 (모집단 모집한다면)
#    - 사용자가 영화 평점을 줬을 때(테스트셋) 게임 추천의 정합도를 평가하려면
#      실제 사용자-게임 상호작용(예: play_count, purchase) 이 필요.
#    - 여기서는 예시 코드(데이터 있을 때 사용)
# =============================
# def evaluate_users(ratings_df, users_list, top_k=10, min_rating=4.0):
#     metrics = {"precision":[], "recall":[], "ndcg":[]}
#     for uid in users_list:
#         # 실제로 사용자가 좋아한 게임(ground truth) 로딩 필요
#         # ground_truth_games = ...
#         # recs = recommend_with_mapper(uid, ratings_df, movie_embs, game_embs, model, top_k=top_k, min_rating=min_rating)
#         # rec_idx = [i for i,_ in recs]
#         # compute precision/recall/ndcg...
#         pass

print("\nnotebook finished. 모델 경로:", MODEL_SAVE)

Sample user: 1
Top recommendations (game_idx, score): [(1516, 0.9532017025720926), (4181, 0.9520097571540259), (1107, 0.9519251899386298), (2266, 0.951332273694719), (749, 0.950015768328295)]
추천 게임 제목 리스트:
Goblins Never Die (Score: 0.953202)
Race With Ryan (Score: 0.952010)
Another Day Off (Score: 0.951925)
Valley Of The Moon (Score: 0.951332)
Treasures of the Roman Empire (Score: 0.950016)

notebook finished. 모델 경로: /content/drive/MyDrive/2025Bigdata/models/recommend_system.pth


In [10]:
# =========================================
# 1) 환경 설정 및 Gradio 설치
# =========================================
# 이미 앞선 코드 셀에서 필요한 라이브러리 (numpy, pandas, torch, sklearn 등)는 임포트되었다고 가정합니다.
# Gradio 설치 (Colab 환경에서)
!pip install gradio -qq

import gradio as gr

# =========================================
# 2) 데이터 로드 및 모델 설정 (기존 코드 재사용)
# =========================================
import os, random, math, re
import numpy as np
import pandas as pd
import ast
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import load_npz

# 1) 경로 설정
MOVIES_CSV = "/content/drive/MyDrive/2025Bigdata/data/movie_dataset/movies_edit.csv"
MOVIE_EMB  = "/content/drive/MyDrive/2025Bigdata/hybrid_movie_embeddings.npy"
RATINGS_CSV = "/content/drive/MyDrive/2025Bigdata/data/movie_dataset/ratings.csv"

# 게임 데이터
GAMES_CSV  = "/content/drive/MyDrive/2025Bigdata/data/game_dataset/preprocessed_steam_data.csv"
GAME_EMB   = "/content/drive/MyDrive/2025Bigdata/data/game_dataset/game_embeddings.npz"
GAME_IDS   = "/content/drive/MyDrive/2025Bigdata/data/game_dataset/embedding_appids.csv"

MODEL_SAVE = "/content/drive/MyDrive/2025Bigdata/models/recommend_system.pth"

# 2) 환경/장치
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3) 데이터 로드 (영화 임베딩은 L2 정규화가 이미 적용되었다고 가정)
print("Loading data...")
try:
    movies = pd.read_csv(MOVIES_CSV)
    movie_embs = np.load(MOVIE_EMB)
    ratings = pd.read_csv(RATINGS_CSV)

    game_embs  = load_npz(GAME_EMB).toarray() # Gradio 함수 내에서 numpy 배열로 사용하기 위해 toarray() 호출
    game_meta_df = pd.read_csv(GAMES_CSV)
    game_ids_df = pd.read_csv(GAME_IDS)

    # 임베딩 순서에 맞춰 메타데이터 병합
    games = pd.merge(
        game_ids_df,
        game_meta_df,
        on='appid',
        how='left'
    )
    # L2 정규화 (기존 코드에서 수행됨, 여기서 다시 수행하여 일관성 유지)
    movie_embs = normalize(movie_embs, axis=1)
    game_embs  = normalize(game_embs, axis=1)

except Exception as e:
    print(f"데이터 로드 중 오류 발생: {e}")
    # Colab에서 실행 시 드라이브 마운트 및 경로 확인 필수

# movieId -> movie_index 매핑 준비 (기존 코드 재사용)
if "movie_index" not in ratings.columns:
    if "movieId" in movies.columns and "movieId" in ratings.columns:
        id2idx = {mid: idx for idx, mid in movies["movieId"].items()}
        ratings["movie_index"] = ratings["movieId"].map(id2idx)
    else:
        print("Movie ID 매핑 오류: ratings.csv에 movieId나 movie_index가 없습니다.")

# 10) Mapper 모델 정의 (기존 코드 재사용)
class MovieToGameMapper(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, output_dim=None, dropout=0.2):
        super().__init__()
        if output_dim is None:
            output_dim = input_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        out = self.net(x)
        out = F.normalize(out, dim=1)  # L2 normalize output
        return out

# 모델 인스턴스화 및 가중치 로드
input_dim = movie_embs.shape[1]
output_dim = game_embs.shape[1]
model = MovieToGameMapper(input_dim=input_dim, hidden_dim=512, output_dim=output_dim, dropout=0.2).to(device)

try:
    model.load_state_dict(torch.load(MODEL_SAVE, map_location=device))
    print("Mapper 모델 가중치 로드 완료.")
except Exception as e:
    print(f"Mapper 모델 가중치 로드 오류: {e}")
    print("모델이 학습되지 않았거나 경로가 잘못되었을 수 있습니다. 추천이 제대로 동작하지 않을 수 있습니다.")

# =========================================
# 3) 추천 로직 함수 (Gradio용)
# =========================================

# 12) helper: map batch (numpy) - 기존 코드 재사용
def map_batch_np(X_np, model, device, batch_size=1024):
    model.eval()
    out_list = []
    for i in range(0, X_np.shape[0], batch_size):
        # Sparse Matrix 대응: X_np가 Sparse일 경우 dense로 변환
        if isinstance(X_np, np.ndarray):
             xb = torch.from_numpy(X_np[i:i+batch_size]).float().to(device)
        else: # (예상치 못한 경우지만, 혹시 모를 sparse input 처리)
             xb = torch.from_numpy(X_np[i:i+batch_size].toarray()).float().to(device)

        with torch.no_grad():
            out = model(xb).cpu().numpy()
        out_list.append(out)
    return np.vstack(out_list)

# 13) 사용자 가중 평균 벡터 생성 함수 - 기존 코드 재사용
def user_weighted_vector(user_id, ratings_df, movie_embs, min_rating=3.5):
    user_df = ratings_df[ratings_df["userId"] == user_id]
    liked = user_df[user_df["rating"] >= min_rating]
    if liked.empty:
        return None
    liked = liked.dropna(subset=["movie_index"])
    idxs = liked["movie_index"].astype(int).values
    scores = liked["rating"].values.astype(float)
    embs = movie_embs[idxs]
    weighted = (embs * scores.reshape(-1,1)).sum(axis=0) / (scores.sum() + 1e-8)
    weighted = weighted / (np.linalg.norm(weighted) + 1e-8)
    return weighted

# 14) 추천 실행 함수 (Gradio 인터페이스의 fn으로 사용)
def get_recommendations(user_id, top_k=5, min_rating=4.0):
    try:
        user_id = int(user_id)
        top_k = int(top_k)
        min_rating = float(min_rating)
    except ValueError:
        return "오류: 사용자 ID, 추천 개수, 최소 평점은 유효한 숫자여야 합니다."

    # 1. 사용자 벡터 생성
    uvec = user_weighted_vector(user_id, ratings, movie_embs, min_rating=min_rating)

    if uvec is None:
        return f"사용자 ID {user_id}에 대해 평점 {min_rating} 이상의 영화 기록이 없습니다. 최소 평점을 낮추거나 다른 사용자를 시도하세요."

    # 2. Mapper 모델을 통해 게임 임베딩 공간으로 투영
    mapped = map_batch_np(uvec.reshape(1,-1), model, device) # normalized

    # 3. 게임 임베딩과의 코사인 유사도 계산
    sims = cosine_similarity(mapped, game_embs).flatten()
    top_idx = sims.argsort()[::-1][:top_k]

    # 4. 결과 포맷팅
    recs = []
    for gi, sc in zip(top_idx, sims[top_idx]):
        if gi < len(games):
            game_name = games.iloc[gi]["original_name"]
            recs.append(f"{game_name} (유사도: {sc:.4f})")
        else:
            recs.append(f"오류: 게임 인덱스 {gi}에 대한 메타데이터를 찾을 수 없습니다.")

    return "\n".join(recs)

# =========================================
# 4) Gradio 인터페이스 정의 및 실행
# =========================================

# 사용 가능한 사용자 ID 예시 찾기
sample_users = ratings["userId"].unique()[:5]
user_examples_fixed = []
for uid in sample_users:
    # 예시 버튼을 누르면: 사용자 ID (uid), Top K (5), 최소 평점 (4.0)을 채우도록 설정
    user_examples_fixed.append([int(uid), 5, 4.0])

# Gradio Interface 정의
iface = gr.Interface(
    fn=get_recommendations,
    inputs=[
        gr.Number(
            label="사용자 ID (userId)",
            value=sample_user,
            interactive=True,
            minimum=1,
            step=1
        ),
        gr.Slider(
            label="추천할 게임 개수 (Top K)",
            minimum=1,
            maximum=20,
            step=1,
            value=5
        ),
        gr.Slider(
            label="영화 '좋아요' 최소 평점 (min_rating)",
            minimum=1.0,
            maximum=5.0,
            step=0.5,
            value=4.0
        )
    ],
    outputs=gr.Textbox(
        label=f"추천 게임 목록 (게임명, 유사도)",
        lines=10,
        placeholder="결과가 여기에 표시됩니다."
    ),
    title="영화 선호도를 기반으로 한 게임 추천 시스템",
    description="Movielens 영화 평점을 바탕으로 사용자 벡터를 생성하고, 학습된 Mapper 모델을 통해 Steam 게임을 추천합니다.",
    examples=user_examples_fixed
)

# Colab에서 실행 시 `share=True`를 사용하면 공개 링크를 생성하여 접근 가능 (약 72시간)
print("\nLaunching Gradio App...")
iface.launch(share=True)

Loading data...
Mapper 모델 가중치 로드 완료.

Launching Gradio App...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6529b32ac21fa4223f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
